# Glossary Quality Assessment

**1. Import and Load Glossary**

In [ ]:
import os
from collections import defaultdict
from glossary_checker import (
    load_yaml, get_slug_line_map, format_line_info, validate_glossary, check_cross_language_links, check_slug_order,
    check_language_order, check_ref_validity, check_def_not_empty, check_def_style, FILE_PATH
)

# Load glossary
glossary = load_yaml(FILE_PATH)
if not glossary:
    print("Failed to load glossary, please check the file path and format.")
else:
    print(f"The glossary contains {len(glossary)} entries.")


**2. Basic Format Validation**

In [ ]:
slug_lines = get_slug_line_map(glossary)
validate_issues = validate_glossary(glossary, slug_lines=slug_lines)
if validate_issues:
    print("\nFound basic format issues:")
    for issue in validate_issues:
        print(f"- {issue}")
else:
    print("\nAll entries comply with basic structure specifications!")


**3.Reference validity check**

In [ ]:
ref_validity_issues = check_ref_validity(glossary, slug_lines=slug_lines)
if ref_validity_issues:
    print("\nIssues found with refs:")
    for issue in ref_validity_issues:
        print(f"- {issue}")
else:
    print("\nAll refs reference valid slugs.")


**4.Reference Consistency Check**

In [ ]:
cross_link_issues = check_cross_language_links(glossary)

if cross_link_issues:
    grouped = defaultdict(list)
    for issue in cross_link_issues:
        line = slug_lines.get(issue['slug']) if slug_lines else None
        key = (issue['slug'], line, issue['missing_link'])
        grouped[key].append(issue['lang'])

    lang_dict = defaultdict(list)
    for (slug, line, missing_link), langs in grouped.items():
        for lang in langs:
            lang_dict[lang].append((slug, line, missing_link))
    
    print("\nMissing cross-reference links found (grouped by language):")
    for lang in sorted(lang_dict.keys()):
        print(f"\n{lang}:")
        for slug, line, missing_link in sorted(lang_dict[lang], key=lambda x: (x[0], x[1] or 0)):
            line_info = f"line {line}" if line else ""
            print(f"  slug '{slug}' ({line_info}) missing link #{missing_link}")
else:
    print("\nAll cross-reference links are consistent across languages.")

**5. Slug Order Check**  

In [ ]:
slug_order_issues = check_slug_order(glossary, slug_lines=slug_lines)

if slug_order_issues:
    print("\nEntries are not sorted in slug alphabetical order:")
    for issue in slug_order_issues:
        print(f"- {issue}")
else:
        print("\nEntries are sorted in slug alphabetical order.")


**6. Language Code Order Check**

In [ ]:
language_order_issues = check_language_order(glossary, slug_lines=slug_lines)

if language_order_issues:
    print("\nEntries with incorrect language code order:")
    for issue in language_order_issues:
        print(f"- {issue}")
else:
    print("\nLanguage Codes in all entries are sorted alphabetically.")


**7. Def Fields Non-empty check**

In [ ]:
empty_def_issues = check_def_not_empty(glossary, slug_lines=slug_lines)
if empty_def_issues:
    print("\nIssues found with def fields:")
    for issue in empty_def_issues:
        print(f"- {issue}")
else:
    print("\nAll def fields are non-empty.")

**8. Def Fields Format check**

In [ ]:
style_issues = check_def_style(glossary, '>', slug_lines=slug_lines)
if style_issues:
    print("\nYAML style issues found in def fields:")
    for issue in style_issues:
        print(f"- {issue}")
else:
    print("\nAll def fields use the correct YAML folded style.")